# Exploratory Data Analysis — Claims Severity

This notebook runs entirely on **synthetic** data generated by `src.data.make_smoke_data`, so it works out of the box with no Kaggle download and no real/sensitive claims data. Swap in `src.data.load_data('data/train.csv')` once you've placed a licensed dataset in `data/` (see `data/README.md`).

Flow: load data → inspect missingness/cardinality → target distribution (raw and log1p) → train/test distribution comparison → categorical frequency → leakage checks → pointers to `src.train` outputs.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import make_smoke_data

pd.set_option('display.max_columns', 50)
df = make_smoke_data(n=2000, seed=42)
df.head()

## 1. Shape, dtypes, and missingness

In [ ]:
print(df.shape)
print(df.dtypes)
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0]

## 2. Cardinality of categorical variables

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    print(c, '->', df[c].nunique(), 'unique values')
    print(df[c].value_counts(normalize=True).round(3))
    print()

## 3. Target (`loss`) distribution: raw vs. log1p

Claim severity is typically right-skewed; a log1p transform (used in `src.train`) makes it closer to symmetric and stabilizes RMSLE-style evaluation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df['loss'], bins=40)
axes[0].set_title('Raw loss distribution')
axes[1].hist(np.log1p(df['loss']), bins=40)
axes[1].set_title('log1p(loss) distribution')
plt.tight_layout()
plt.show()

df['loss'].describe()

## 4. Train/test distribution comparison

A quick sanity check that a random split doesn't introduce obvious covariate shift. For real claims data, prefer a time-based or claimant-based split if timestamps or repeat claimants exist, to avoid leakage.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

for col in ['age', 'vehicle_age']:
    print(col, 'train mean:', round(train_df[col].mean(), 2), '| test mean:', round(test_df[col].mean(), 2))

for col in cat_cols:
    train_freq = train_df[col].value_counts(normalize=True).round(3)
    test_freq = test_df[col].value_counts(normalize=True).round(3)
    compare = pd.DataFrame({'train': train_freq, 'test': test_freq})
    print(col)
    print(compare)
    print()

## 5. Feature vs. target relationships

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(df['age'], np.log1p(df['loss']), alpha=0.2, s=8)
axes[0].set_xlabel('age'); axes[0].set_ylabel('log1p(loss)')
axes[1].scatter(df['vehicle_age'], np.log1p(df['loss']), alpha=0.2, s=8)
axes[1].set_xlabel('vehicle_age'); axes[1].set_ylabel('log1p(loss)')
plt.tight_layout()
plt.show()

df.groupby('state')['loss'].mean().sort_values(ascending=False)

## 6. Leakage checks (manual notes)

- No feature here is derived from the target (`loss`) itself.
- No claimant/policy identifier is present that could leak across a train/test split.
- For real data: check for post-event fields (e.g. payout date, adjuster notes written after settlement) that would not be available at prediction time, and drop or lag them.
- `src/validate.py` enforces a schema/data contract to catch structural issues (missing columns, out-of-range values) before this stage.

## 7. Next steps

Run training and inspect metrics:

```bash
python -m src.train                 # synthetic smoke data
python -m src.train --data data/train.csv --target loss  # real data, if available
mlflow ui --backend-store-uri mlruns
```

See `docs/interview_guide.md` for metric definitions (MAE/RMSE/RMSLE) and modeling trade-offs, and `docs/responsible_ai.md` for the responsible-AI safeguards that apply once predictions leave this notebook.